# Tahap 2 — Case Representation
## CBR Putusan Wanprestasi PN Surabaya

**Input:**
- `data/raw/*.txt` → teks bersih hasil Tahap 1 (sudah lowercase, tanda baca → spasi)
- `data/processed/metadata_raw.csv` → metadata dari Cell 10 Tahap 1

**Output:**
- `data/processed/cases.csv`
- `data/processed/cases.json`

**Kolom output sesuai soal:**

| case_id | no_perkara | tanggal | ringkasan_fakta | pasal | pihak | argumen_hukum | amar_putusan | word_count | top_terms | label_putusan | text_full |
|---|---|---|---|---|---|---|---|---|---|---|---|

## Setup - Jangkar Working Directory ke Root Repository

**Jalankan cell ini SELALU sebagai cell pertama**, sebelum cell lain di notebook ini.

Notebook ini disimpan di dalam folder `notebooks/`, tapi seluruh path data di notebook ini
(mis. `'data/processed/cases.csv'`) ditulis **relatif terhadap root repository**, bukan terhadap
folder `notebooks/`. Secara default, Jupyter menjalankan kernel dengan *working directory* = folder
tempat file `.ipynb` itu sendiri berada. Tanpa cell ini, path seperti `'data/raw'` akan dicari di
`notebooks/data/raw` (tidak ada) dan menyebabkan `FileNotFoundError`.

Cell ini **aman dijalankan berkali-kali** (idempotent) — begitu working directory sudah berada di
root repository (folder yang punya subfolder `data/`), cell ini tidak akan berpindah lagi.

In [55]:
import os

# Jika folder 'data/' tidak ada di working directory saat ini, TAPI ada satu
# level di atasnya -> kita sedang di dalam notebooks/, pindah ke root repo.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')

print('Working directory aktif :', os.getcwd())
print('Isi folder saat ini     :', sorted(os.listdir('.')))

assert os.path.isdir('data'), (
    "Folder 'data/' tidak ditemukan dari working directory ini.\n"
    "Pastikan struktur repo: <root>/notebooks/xx.ipynb dengan <root>/data/ di sebelahnya,\n"
    "dan notebook dibuka dari dalam struktur repo tersebut (bukan dipindah sendirian)."
)


Working directory aktif : c:\Users\LENOVO\Documents\Penalaran Komputer\Tugas_3\Tugas3.1
Isi folder saat ini     : ['.git', '.gitignore', 'README.md', 'data', 'logs', 'notebooks', 'requirements.txt']


## Cell 1 — Import Library

In [56]:
import os
import re
import json
import pandas as pd
from collections import Counter

print('Library siap')
print(f'pandas versi : {pd.__version__}')

Library siap
pandas versi : 2.3.3


## Cell 2 — Konfigurasi Path & Load Metadata Tahap 1

In [57]:
DIR_TXT       = 'data/raw'
DIR_PROCESSED = 'data/processed'
PATH_META_RAW = 'data/processed/metadata_raw.csv'
PATH_CSV_OUT  = 'data/processed/cases.csv'
PATH_JSON_OUT = 'data/processed/cases.json'

os.makedirs(DIR_PROCESSED, exist_ok=True)

# Load metadata dari Tahap 1
df_meta = pd.read_csv(PATH_META_RAW, encoding='utf-8-sig')

print(f'metadata_raw.csv dimuat : {len(df_meta)} baris')
print(f'Kolom tersedia          : {list(df_meta.columns)}')
print()
df_meta[['case_id', 'no_perkara', 'tanggal', 'amar_putusan']].head(5)

metadata_raw.csv dimuat : 54 baris
Kolom tersedia          : ['case_id', 'no_perkara', 'tanggal', 'jenis_perkara', 'pengadilan', 'pihak', 'pasal', 'amar_putusan', 'url_sumber', 'path_pdf']



,case_id,no_perkara,tanggal,amar_putusan
0,case_001,1124/Pdt.G/2024/PN Sby,21 Oktober 2024,Sengketa Kewenangan Mengadili
1,case_002,1225/Pdt.G/2024/PN Sby,18 Nopember 2024,Sengketa Kewenangan Mengadili
2,case_003,75/Pdt.G/2025/PN Sby,16 Januari 2025,Sengketa Kewenangan Mengadili
3,case_004,1223/Pdt.G/2024/PN Sby,18 Nopember 2024,Sengketa Kewenangan Mengadili
4,case_005,680/Pdt.G/2024/PN Sby,1 Juli 2024,Sengketa Kewenangan Mengadili


## Cell 3 — Fungsi Ekstraksi Konten dari Teks

Teks dari Tahap 1 sudah **lowercase** dan tanda baca sudah diganti spasi.  
Semua pola regex menggunakan huruf kecil.

Tiga bagian yang diekstrak:
1. **Ringkasan fakta** — bagian `tentang duduk perkara`
2. **Argumen hukum** — bagian `pertimbangan hukum`
3. **Amar putusan lengkap** — bagian `mengadili`

In [58]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 (PERBAIKAN FINAL) — Fungsi Ekstraksi Konten dari Teks
# Menggabungkan SEMUA fungsi yang dibutuhkan Cell 5 (Cell 14):
#   - ekstrak_ringkasan_fakta   (REKONSTRUKSI — sempat hilang)
#   - ekstrak_argumen_hukum     (REKONSTRUKSI — sempat hilang)
#   - ekstrak_amar_lengkap      (versi robust, toleran spasi-antar-huruf)
#   - derive_label              (versi robust, dengan fallback teks_full)
#   - ekstrak_pihak_dari_teks
#   - ekstrak_pasal_dari_teks
#   - gabung_huruf_berspasi     (fungsi bantu)
# ═══════════════════════════════════════════════════════════════════════

def gabung_huruf_berspasi(teks: str) -> str:
    teks = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                  lambda m: m.group().replace(' ', ''), teks)

    # PERBAIKAN: tangani 'm e l a w a n' yang menempel langsung ke kata
    # berikutnya tanpa spasi (mis. '...a nwilliam'), sehingga huruf 'n'
    # terakhir gagal ikut tergabung oleh pola umum di atas.
    teks = re.sub(r'\bmelawa\s*n(?=[a-z])', 'melawan ', teks)
    teks = re.sub(r'\blawa\s*n(?=[a-z])', 'lawan ', teks)

    return teks


def ekstrak_ringkasan_fakta(teks: str, max_chars: int = 1500) -> str:
    """
    Ekstrak ringkasan fakta perkara dari bagian 'TENTANG DUDUK PERKARA' /
    posita gugatan, yaitu uraian kronologis kejadian sebelum bagian
    pertimbangan hukum dimulai.
    """
    pola_mulai = [
        r'tentang duduk perkara',
        r'duduk perkara',
        r'menimbang bahwa penggugat dengan surat gugatan',
    ]
    pola_selesai = [
        r'tentang pertimbangan hukum',
        r'menimbang bahwa maksud dan tujuan gugatan',
        r'dalam eksepsi',
        r'dalam pokok perkara',
    ]

    idx_mulai = -1
    for pola in pola_mulai:
        m = re.search(pola, teks)
        if m:
            idx_mulai = m.end()
            break

    if idx_mulai == -1:
        # Fallback: ambil bagian awal dokumen jika penanda baku tidak ditemukan
        return teks[:max_chars].strip()

    idx_selesai = len(teks)
    for pola in pola_selesai:
        m = re.search(pola, teks[idx_mulai:])
        if m:
            idx_selesai = idx_mulai + m.start()
            break

    return teks[idx_mulai:idx_selesai].strip()[:max_chars]


def ekstrak_argumen_hukum(teks: str, max_chars: int = 1500) -> str:
    """
    Ekstrak argumen hukum utama dari bagian 'TENTANG PERTIMBANGAN HUKUM'
    sampai sebelum bagian amar (MENGADILI).
    """
    pola_mulai = [
        r'tentang pertimbangan hukum',
        r'menimbang bahwa maksud dan tujuan gugatan',
    ]
    pola_selesai = [
        r'mengadili',
        r'm e n g a d i l i',
    ]

    idx_mulai = -1
    for pola in pola_mulai:
        m = re.search(pola, teks)
        if m:
            idx_mulai = m.end()
            break

    if idx_mulai == -1:
        return ''

    idx_selesai = len(teks)
    for pola in pola_selesai:
        matches = list(re.finditer(pola, teks[idx_mulai:]))
        if matches:
            idx_selesai = idx_mulai + matches[-1].start()
            break

    return teks[idx_mulai:idx_selesai].strip()[:max_chars]


def ekstrak_amar_lengkap(teks: str, max_chars: int = 800) -> str:
    """
    Ekstrak bunyi amar putusan lengkap.
    Ambil kemunculan 'mengadili' yang TERAKHIR karena biasanya di akhir dokumen.
    Digabung-huruf-berspasi dulu agar pola 'm e n g a d i l i' dengan
    spasi tak rata tetap terdeteksi.
    """
    teks_gabung = gabung_huruf_berspasi(teks)

    pola_mulai = [r'mengadili']
    pola_selesai = [r'demikian diputuskan', r'perincian biaya perkara', r'hakim ketua']

    idx_mulai = -1
    for pola in pola_mulai:
        matches = list(re.finditer(pola, teks_gabung))
        if matches:
            idx_mulai = matches[-1].start()
            break

    if idx_mulai == -1:
        # Fallback terakhir: ambil 800 karakter TERAKHIR dokumen
        # (amar putusan selalu berada di bagian penutup dokumen)
        return teks_gabung[-max_chars:].strip()

    idx_selesai = len(teks_gabung)
    for pola in pola_selesai:
        m = re.search(pola, teks_gabung[idx_mulai:])
        if m:
            idx_selesai = idx_mulai + m.start()
            break

    return teks_gabung[idx_mulai:idx_selesai].strip()[:max_chars]


def derive_label(amar_meta: str, amar_teks: str, teks_full: str = '') -> int:
    """
    Tentukan label putusan: 1=Dikabulkan, 0=Ditolak/NO, 2=Damai/Penetapan,
    3=Masih Proses, -1=Unknown.
    Daftar kata kunci diperluas + fallback ke teks_full kalau amar_teks kosong/tak cocok.
    """
    teks = str(amar_teks).lower()

    kata_kabul = [
        'dikabulkan', 'mengabulkan', 'menghukum tergugat',
        'menyatakan tergugat telah wanprestasi',
        'mengabulkan gugatan', 'dikabulkan untuk sebagian',
        'dikabulkan sebagian', 'mengabulkan sebagian',
    ]
    kata_tolak = [
        'ditolak', 'menolak', 'niet ontvankelijk', 'niet ontvankelijke',
        'gugur', 'tidak dapat diterima', 'menghukum penggugat',
        'menolak gugatan', 'gugatan penggugat tidak dapat diterima',
    ]
    kata_damai = [
        'akta perdamaian', 'perdamaian', 'mencabut gugatan',
        'pencabutan', 'dicabut',
    ]
    kata_proses = [
        'melanjutkan pemeriksaan', 'menangguhkan biaya',
    ]

    def cek(t):
        # Kabul/tolak = sinyal definitif dari diktum vonis -> cek DULUAN
        for k in kata_kabul:
            if k in t:
                return 1
        for k in kata_tolak:
            if k in t:
                return 0
        # Damai/proses dicek belakangan, karena kata-katanya rawan ambigu/boilerplate
        for k in kata_damai:
            if k in t:
                return 2
        for k in kata_proses:
            if k in t:
                return 3
        return -1

    label = cek(teks)
    if label != -1:
        return label

    # Fallback: kalau amar_teks kosong/tidak ada kata kunci, coba cek di
    # SELURUH isi dokumen (teks_full)
    if teks_full:
        teks_full_gabung = gabung_huruf_berspasi(str(teks_full).lower())
        return cek(teks_full_gabung)

    return -1


def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di tengah kalimat akibat page-break PDF
    teks_norm = re.sub(
        r'halaman\s*\d+\s*(?:dari\s*\d+\s*)?putusan\s*no\.?\s*(?:mor)?\s*\d+\s*pdt\s*g\s*\d{4}\s*pn\s*sby',
        ' ', teks_norm, flags=re.IGNORECASE
    )

    nama_bulan = r'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'

    pemutus = (
        r'(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|'
        r'\s+dalam\s+kapasitas|\s+bertindak|\s+berdasarkan|\s+yang\s*berkedudukan|\s+berkedudukan|'
        r'\s+diwakili|\s+lawan|\s+melawan|\s+tempat\s+tanggal\s*lahir|\s+tempat\s+dan\s+tanggal\s*lahir|'
        r'\s+tempat\s+tgl\s*lahir|\s+tempat\s+lahir|\s+lahir\s+di|\s+lahir\s+pada|\s+umur|\s+jenis\s+kelamin|'
        r'\s+bertempat\s+tinggal|\s+kewarganegaraan|\s+warga\s+negara|\s+yang\b|\s+selaku|'
        r'\s+dalam\s+jabatannya|\s+alias|\s+cq|\s+no\s+identitas|\s+seorang|\s+sebagai\s+tergugat|'
        r'\s+sebagai\s+penggugat|\s+istri|\s+suami|\s+dahulu|\s+halaman|\s+putusan\s+nomor|\s+putusan\s+no|'
        r'\s+\d{1,2}\s+(?:' + nama_bulan + r')\s+\d{4})'  # PERBAIKAN BARU: pola tanggal langsung
    )

    def rapikan_nama(raw):
        kata = raw.strip().split()
        return ' '.join(kata[:8]).title()

    penggugat = 'Tidak Tertera'
    m = re.search(
        r'perkara\s*(?:gugatan\s*)?antara\s+([a-z0-9][a-z0-9\s\.]{2,200}?)' + pemutus,
        teks_norm, re.IGNORECASE
    )
    if m:
        penggugat = rapikan_nama(m.group(1))

    tergugat = 'Tidak Tertera'
    m = re.search(
        r'\b(?:me)?lawan\s+(?!hukum|logika|bantahan|perlawanan)([a-z0-9][a-z0-9\s\.]{2,200}?)' + pemutus,
        teks_norm, re.IGNORECASE
    )
    if m:
        tergugat = rapikan_nama(m.group(1))

    return f'{penggugat} vs. {tergugat}'

def ekstrak_pasal_dari_teks(teks: str) -> str:
    """
    Ekstrak referensi pasal langsung dari teks putusan (sudah lowercase).
    Menangkap pola: 'pasal 1234', 'pasal 1234 ayat 1', 'pasal 1234 jo 1235'
    Contoh hasil: 'Pasal 1243, Pasal 1320, Pasal 1338'
    """
    matches = re.findall(
        r'pasal\s+(\d+(?:\s*(?:ayat|huruf|dan|jo\.?|juncto|junto)?\s*\d*)*)',
        teks, re.IGNORECASE
    )
    pasal_list = []
    seen = set()
    for m in matches:
        nomor = re.sub(r'\s+', ' ', m.strip())
        if re.search(r'\d{3,}', nomor):
            label = f'Pasal {nomor}'
            if label not in seen:
                pasal_list.append(label)
                seen.add(label)
    return ', '.join(pasal_list[:15]) if pasal_list else 'Tidak Tertera'


# --- Test ulang pada case_002 sebagai verifikasi seluruh fungsi tersedia ---
print('Semua fungsi ekstraksi siap (8 fungsi):')
print('  - gabung_huruf_berspasi, ekstrak_ringkasan_fakta, ekstrak_argumen_hukum,')
print('  - ekstrak_amar_lengkap, derive_label, ekstrak_pihak_dari_teks, ekstrak_pasal_dari_teks')
print()

sample_path = os.path.join(DIR_TXT, 'case_002.txt')
if os.path.exists(sample_path):
    with open(sample_path, 'r', encoding='utf-8') as f:
        sample_teks = f.read()

    rf  = ekstrak_ringkasan_fakta(sample_teks)
    ah  = ekstrak_argumen_hukum(sample_teks)
    al  = ekstrak_amar_lengkap(sample_teks)
    lbl = derive_label('', al, sample_teks)
    pihak = ekstrak_pihak_dari_teks(sample_teks)
    pasal = ekstrak_pasal_dari_teks(sample_teks)

    print(f'--- TEST case_002 ---')
    print(f'ringkasan_fakta ({len(rf)} char): {rf[:150]}...')
    print(f'argumen_hukum   ({len(ah)} char): {ah[:150]}...')
    print(f'amar_lengkap    ({len(al)} char): {al[:150]}...')
    print(f'pihak           : {pihak}')
    print(f'pasal           : {pasal}')
    print(f'label_putusan   : {lbl}')
else:
    print('PERINGATAN: case_002.txt tidak ditemukan. Pastikan Tahap 1 sudah selesai.')


Semua fungsi ekstraksi siap (8 fungsi):
  - gabung_huruf_berspasi, ekstrak_ringkasan_fakta, ekstrak_argumen_hukum,
  - ekstrak_amar_lengkap, derive_label, ekstrak_pihak_dari_teks, ekstrak_pasal_dari_teks

--- TEST case_002 ---
ringkasan_fakta (1500 char): menimbang bahwa penggugat dengan surat gugatan tanggal 11nopember 2024 yang diterima dan didaftarkan di kepaniteraan pengadilannegeri surabaya pada ta...
argumen_hukum   (1500 char): menimbang bahwa maksud dan tujuan gugatan penggugat yang padapokoknya adalah sebagaimana tersebut di atas dalam konpensi dalam eksepsi

menimbang bahw...
amar_lengkap    (404 char): mengadili dalam konpensi dalam eksepsi menerima eksepsi dari tergugat dalam pokok perkara menyatakan gugatan penggugat tidak dapat diterima niet ontva...
pihak           : Kandar Panjaitan vs. Johan Sugiharto Se
pasal           : Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332
label_putusan   : 0


In [59]:
# Cek urutan eksekusi cell di notebook ini
print("Apakah ada error di cell sebelumnya? Scroll ke atas dan cari tanda [*] yang macet,")
print("atau cell dengan kotak merah/traceback error.")

Apakah ada error di cell sebelumnya? Scroll ke atas dan cari tanda [*] yang macet,
atau cell dengan kotak merah/traceback error.


In [60]:
# Debug: cek detail kegagalan pada beberapa case spesifik
for cid in ['case_039', 'case_006', 'case_017']:
    with open(f'data/raw/{cid}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()

    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                        lambda m: m.group().replace(' ', ''), teks)

    print(f"=== {cid} ===")
    # Cari posisi "antara" dan "lawan" di teks yang sudah dinormalisasi
    m_antara = re.search(r'perkara\s+(?:gugatan\s+)?antara', teks_norm, re.IGNORECASE)
    m_lawan = re.search(r'\b(?:me)?lawan\b', teks_norm, re.IGNORECASE)

    if m_antara:
        print(f"'perkara...antara' ditemukan di posisi {m_antara.start()}")
        print(f"  Konteks: ...{teks_norm[m_antara.start():m_antara.start()+250]}...")
    else:
        print("'perkara...antara' TIDAK ditemukan setelah normalisasi!")

    print()
    if m_lawan:
        print(f"'lawan' ditemukan di posisi {m_lawan.start()}")
        print(f"  Konteks: ...{teks_norm[m_lawan.start():m_lawan.start()+250]}...")
    else:
        print("'lawan' TIDAK ditemukan setelah normalisasi!")
    print()
    print("-" * 70)

=== case_039 ===
'perkara...antara' ditemukan di posisi 228
  Konteks: ...perkara antara santi boediyanto tempat tanggal lahir surabaya 13 09 1968 umur 56 tahun jenis kelamin perempuan agama islam warga negara indonesia alamat apartemen rose baya 902 rt 004 rw 002 kel pradah kalikendal kec dukuh pakis surabaya pekerjaaan w...

'lawan' ditemukan di posisi 1029
  Konteks: ...lawan 1 go gorry lisayati tempat tanggal lahir surabaya o8 02 1978 umur 46 tahun jenis kelamin perempuan agama islam warga negara indonesia alamat jl tanjungtorawitan no 23 25 rt 006 rw 008 kel perak barat kec krembangan surabaya pekerjaaan mengurusr...

----------------------------------------------------------------------
=== case_006 ===
'perkara...antara' ditemukan di posisi 236
  Konteks: ...perkara antara daniel douglas wijaya tempat tgl lahir surabaya 05 januari 1992 jenis kelamin laki laki agama budha warganegara indonesia alamat jl yupiter bs 14 rt 010rw 003 kel tanjungsarin kec sukomanunggal kotasurabaya da

In [61]:
with open('data/raw/case_039.txt', 'r', encoding='utf-8') as f:
    teks_039 = f.read()

hasil = ekstrak_pihak_dari_teks(teks_039)
print("Hasil fungsi:", hasil)

Hasil fungsi: Santi Boediyanto vs. 1 Go Gorry Lisayati


In [62]:
with open('data/raw/case_039.txt', 'r', encoding='utf-8') as f:
    teks_039 = f.read()

# Langkah 1: cek HASIL NORMALISASI (gabung_huruf_berspasi) pada teks penuh
teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                    lambda m: m.group().replace(' ', ''), teks_039)

# Langkah 2: cari "perkara...antara" di teks yang SUDAH dinormalisasi (bukan teks asli)
m_antara = re.search(r'perkara\s+(?:gugatan\s+)?antara', teks_norm, re.IGNORECASE)
print("Match 'perkara...antara' pada teks_norm:", m_antara)

if m_antara:
    print()
    print("Konteks 300 karakter setelah match:")
    print(teks_norm[m_antara.start():m_antara.start()+300])
else:
    print()
    print("TIDAK DITEMUKAN. Mari bandingkan dengan teks ASLI (sebelum normalisasi):")
    m_antara_asli = re.search(r'perkara\s+(?:gugatan\s+)?antara', teks_039, re.IGNORECASE)
    print("Match pada teks ASLI:", m_antara_asli)
    if m_antara_asli:
        print("Konteks teks ASLI:", teks_039[m_antara_asli.start():m_antara_asli.start()+300])
        print()
        print("Konteks teks_norm di posisi yang SAMA (untuk lihat apa yang berubah):")
        print(teks_norm[m_antara_asli.start():m_antara_asli.start()+300])

Match 'perkara...antara' pada teks_norm: <re.Match object; span=(228, 242), match='perkara antara'>

Konteks 300 karakter setelah match:
perkara antara santi boediyanto tempat tanggal lahir surabaya 13 09 1968 umur 56 tahun jenis kelamin perempuan agama islam warga negara indonesia alamat apartemen rose baya 902 rt 004 rw 002 kel pradah kalikendal kec dukuh pakis surabaya pekerjaaan wiraswasta status cerai hidup pendidikan sma dalam 


In [63]:
import inspect
print(inspect.getsource(ekstrak_pihak_dari_teks))

def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di tengah kalimat akibat page-break PDF
    teks_norm = re.sub(
        r'halaman\s*\d+\s*(?:dari\s*\d+\s*)?putusan\s*no\.?\s*(?:mor)?\s*\d+\s*pdt\s*g\s*\d{4}\s*pn\s*sby',
        ' ', teks_norm, flags=re.IGNORECASE
    )

    nama_bulan = r'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'

    pemutus = (
        r'(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|'
        r'\s+dalam\s+kapasitas|\s+bertindak|\s+berdasarkan|\s+yang\s*berkedudukan|\s+berkedudukan|'
        r'\s+diwakili|\s+lawa

In [64]:
import inspect
print(inspect.getsource(ekstrak_pihak_dari_teks))

def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di tengah kalimat akibat page-break PDF
    teks_norm = re.sub(
        r'halaman\s*\d+\s*(?:dari\s*\d+\s*)?putusan\s*no\.?\s*(?:mor)?\s*\d+\s*pdt\s*g\s*\d{4}\s*pn\s*sby',
        ' ', teks_norm, flags=re.IGNORECASE
    )

    nama_bulan = r'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'

    pemutus = (
        r'(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|'
        r'\s+dalam\s+kapasitas|\s+bertindak|\s+berdasarkan|\s+yang\s*berkedudukan|\s+berkedudukan|'
        r'\s+diwakili|\s+lawa

In [65]:
for cid in ['case_023', 'case_045']:
    with open(f'data/raw/{cid}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()

    idx = teks.lower().find('lawan')
    print(f"=== {cid} ===")
    if idx != -1:
        print(teks[max(0,idx-20):idx+250])
    else:
        print("Kata 'lawan' TIDAK DITEMUKAN sama sekali dalam dokumen ini!")
    print()

=== case_023 ===
aad meskipun ada perlawanan banding kasasi maupun verzet 12 menghukum tergugat untuk tunduk dan patuh pada putusan ini 13 menghukum tergugat untuk membayar biaya perkara ini subsidair dalam peradilan yang baik mohon putusan yang seadil adilnya menurut hukumdan kebenaran

=== case_045 ===
aad meskipun ada perlawananbanding kasasi maupun verzet atauapabila majelis hakim yang memeriksa mengadili memutus perkara a quo berpendapat lain mohon putusan yang seadil adilnya ex aequo et bono menimbang bahwa pada hari persidangan yang telah ditetapkan penggugat dan



## Cell 4 — Feature Engineering

Sesuai soal: hitung `length` (jumlah kata) dan `bag-of-words` sederhana.

In [66]:
# Stopwords — sama dengan yang dipakai di Tahap 1
STOPWORDS = {
    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'atau',
    'pada', 'dengan', 'adalah', 'juga', 'oleh', 'serta', 'pula',
    'pun', 'nya', 'si', 'tidak', 'telah', 'akan', 'bahwa', 'untuk',
    'dalam', 'atas', 'tersebut', 'para', 'sebagai', 'sudah',
    'hal', 'maka', 'agar', 'karena', 'namun', 'tetapi', 'jika',
    'maka', 'pihak', 'perkara', 'pula', 'maupun',
}


def hitung_word_count(teks: str) -> int:
    """Hitung jumlah kata (length) sesuai permintaan soal."""
    return len(teks.split())


def bag_of_words_top(teks: str, n: int = 10) -> str:
    """
    Bag-of-words sederhana: n kata paling sering muncul.
    Abaikan stopwords dan kata pendek (< 4 huruf).
    Dikembalikan sebagai string dipisah koma untuk disimpan di CSV.
    """
    tokens = teks.split()
    tokens_bersih = [
        t for t in tokens
        if t not in STOPWORDS
        and len(t) >= 4
        and t.isalpha()
    ]
    counter = Counter(tokens_bersih)
    return ', '.join([w for w, _ in counter.most_common(n)])


print('Fungsi feature engineering siap')
print()

if os.path.exists(sample_path):
    print(f'word_count case_002 : {hitung_word_count(sample_teks):,} kata')
    print(f'bag_of_words top-10 : {bag_of_words_top(sample_teks)}')

Fungsi feature engineering siap

word_count case_002 : 9,165 kata
bag_of_words top-10 : penggugat, tergugat, bukti, selanjutnya, fotokopi, diberi, tanda, gugatan, pekerjaan, bangunan


## Cell 5 — Proses Semua Dokumen → Simpan CSV & JSON

In [67]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 (PERBAIKAN) — Proses Semua Dokumen -> Simpan CSV & JSON
# Pengganti Cell 10 lama. Bedanya hanya 1 baris: derive_label() sekarang
# diberi parameter ketiga `teks` (teks_full) sebagai fallback pencarian
# kata kunci, untuk dokumen yang amar_lengkap-nya kosong/tidak cocok pola.
# ═══════════════════════════════════════════════════════════════════════

hasil = []
tidak_ditemukan = []

print('=' * 62)
print('  CASE REPRESENTATION — MEMPROSES SEMUA DOKUMEN')
print('=' * 62)
print()

for _, row in df_meta.iterrows():
    case_id  = row['case_id']
    path_txt = os.path.join(DIR_TXT, f'{case_id}.txt')

    if not os.path.exists(path_txt):
        tidak_ditemukan.append(case_id)
        print(f'  LEWATI: {case_id} — file .txt tidak ditemukan')
        continue

    with open(path_txt, 'r', encoding='utf-8') as f:
        teks = f.read()

    # Ekstraksi konten
    ringkasan_fakta = ekstrak_ringkasan_fakta(teks)
    argumen_hukum   = ekstrak_argumen_hukum(teks)
    amar_lengkap    = ekstrak_amar_lengkap(teks)

    # Feature engineering
    wc  = hitung_word_count(teks)
    bow = bag_of_words_top(teks)

    # Label putusan — sekarang dengan fallback ke teks_full (`teks`)
    label = derive_label(row.get('amar_putusan', ''), amar_lengkap, teks)

    hasil.append({
        'case_id'         : case_id,
        'no_perkara'      : row.get('no_perkara', ''),
        'tanggal'         : row.get('tanggal', ''),
        'jenis_perkara'   : row.get('jenis_perkara', 'Perdata Wanprestasi'),
        'pengadilan'      : row.get('pengadilan', 'PN Surabaya'),
        'pihak'           : ekstrak_pihak_dari_teks(teks),
        'pasal'           : ekstrak_pasal_dari_teks(teks),
        'ringkasan_fakta' : ringkasan_fakta,
        'argumen_hukum'   : argumen_hukum,
        'amar_putusan'    : row.get('amar_putusan', ''),
        'amar_lengkap'    : amar_lengkap,
        'word_count'      : wc,
        'top_terms'       : bow,
        'label_putusan'   : label,
        'text_full'       : teks,
    })

    lbl = {1: 'Dikabulkan', 0: 'Ditolak/NO', 2: 'Damai/Penetapan', 3: 'Masih Proses', -1: 'Unknown'}[label]
    print(f'  OK  {case_id} | {wc:,} kata | {lbl}')


# Buat DataFrame
df_cases = pd.DataFrame(hasil)

# Simpan CSV

# BARU — urutan kolom sesuai soal, text_full di kolom terakhir:
KOLOM_CSV = [
    'case_id', 'no_perkara', 'tanggal', 'ringkasan_fakta',
    'pasal', 'pihak', 'argumen_hukum', 'amar_lengkap',
    'word_count', 'top_terms', 'label_putusan', 'text_full'
]

df_cases[KOLOM_CSV].to_csv(PATH_CSV_OUT, index=False, encoding='utf-8-sig')

print()
print(f'CSV tersimpan  -> {PATH_CSV_OUT}  ({len(df_cases)} baris)')

# Simpan JSON (tanpa text_full agar file tidak terlalu besar)
records = df_cases.drop(columns=['text_full']).to_dict(orient='records')
with open(PATH_JSON_OUT, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f'JSON tersimpan -> {PATH_JSON_OUT}  (tanpa kolom text_full)')

if tidak_ditemukan:
    print(f'PERINGATAN: {len(tidak_ditemukan)} file tidak ditemukan -> {tidak_ditemukan}')

# Ringkasan sisa Unknown setelah perbaikan
sisa_unknown = (df_cases['label_putusan'] == -1).sum()
print()
print(f'Sisa label Unknown setelah perbaikan: {sisa_unknown} dari {len(df_cases)} dokumen')
if sisa_unknown > 0:
    print('Case_id yang masih Unknown:', df_cases[df_cases['label_putusan'] == -1]['case_id'].tolist())
    print('-> Jalankan Cell 14 (debug) untuk lihat isi amar_lengkap-nya secara manual.')


  CASE REPRESENTATION — MEMPROSES SEMUA DOKUMEN

  OK  case_001 | 16,920 kata | Dikabulkan
  OK  case_002 | 9,165 kata | Ditolak/NO
  OK  case_003 | 6,515 kata | Dikabulkan
  OK  case_004 | 41,500 kata | Ditolak/NO
  OK  case_005 | 14,836 kata | Dikabulkan
  OK  case_006 | 8,781 kata | Dikabulkan
  OK  case_007 | 24,445 kata | Ditolak/NO
  OK  case_008 | 16,793 kata | Ditolak/NO
  OK  case_009 | 7,803 kata | Dikabulkan
  OK  case_010 | 9,543 kata | Ditolak/NO
  OK  case_011 | 10,999 kata | Ditolak/NO
  OK  case_012 | 16,870 kata | Dikabulkan
  OK  case_013 | 5,216 kata | Ditolak/NO
  OK  case_014 | 18,565 kata | Ditolak/NO
  OK  case_015 | 11,930 kata | Dikabulkan
  OK  case_016 | 3,620 kata | Ditolak/NO
  OK  case_017 | 4,729 kata | Dikabulkan
  OK  case_018 | 17,950 kata | Dikabulkan
  OK  case_019 | 6,072 kata | Dikabulkan
  OK  case_020 | 3,266 kata | Dikabulkan
  OK  case_021 | 6,560 kata | Dikabulkan
  OK  case_022 | 6,461 kata | Ditolak/NO
  OK  case_023 | 4,220 kata | Dikabulka

# CEK DOKUMEN

In [68]:
# Verifikasi: tampilkan 4 dokumen yang argumen_hukum-nya kosong/pendek
kosong = df_cases[df_cases['argumen_hukum'].str.len() <= 50]
print(f"Total dokumen argumen_hukum kosong/pendek: {len(kosong)}")
print()
for _, row in kosong.iterrows():
    print(f"case_id    : {row['case_id']}")
    print(f"word_count : {row['word_count']:,} kata")
    print(f"label      : {row['label_putusan']}")
    print(f"amar_lengkap (300 char) : {row['amar_lengkap'][:300]}")
    print(f"argumen_hukum isi       : '{row['argumen_hukum']}'")
    print("-" * 60)

Total dokumen argumen_hukum kosong/pendek: 0



In [69]:
# Cek SEMUA dokumen yang sebenarnya "Penetapan", bukan "Putusan"
def cek_jenis_dokumen(case_id, dir_txt='data/raw'):
    path = f"{dir_txt}/{case_id}.txt"
    with open(path, 'r', encoding='utf-8') as f:
        teks = f.read()[:500].lower()  # cek 500 karakter pertama saja
    if 'telah menjatuhkan penetapan' in teks or 'telah memberikan penetapan' in teks:
        return 'PENETAPAN'
    elif 'telah menjatuhkan putusan' in teks or 'telah memberikan putusan' in teks:
        return 'PUTUSAN'
    return 'TIDAK_TERDETEKSI'

df_cases['jenis_dokumen'] = df_cases['case_id'].apply(cek_jenis_dokumen)
print(df_cases['jenis_dokumen'].value_counts())
print()
print(df_cases[df_cases['jenis_dokumen'] == 'PENETAPAN'][['case_id', 'word_count', 'label_putusan']])

jenis_dokumen
PUTUSAN             40
TIDAK_TERDETEKSI    14
Name: count, dtype: int64

Empty DataFrame
Columns: [case_id, word_count, label_putusan]
Index: []


In [70]:
# Sample 5 dokumen TIDAK_TERDETEKSI untuk dicek manual
sample_cek = df_cases[df_cases['jenis_dokumen'] == 'TIDAK_TERDETEKSI']['case_id'].head(5)
for cid in sample_cek:
    with open(f"data/raw/{cid}.txt", 'r', encoding='utf-8') as f:
        print(f"=== {cid} ===")
        print(f.read()[:400])
        print()

=== case_006 ===
p u t u s a nnomor 851 pdt g 2024 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang memeriksa dan mengadili perkara perdatagugatan dalam peradilan tingkat pertama menjatuhkan putusan sebagai berikutdalam perkara antara daniel douglas wijaya tempat tgl lahir surabaya 05 januari 1992 jenis kelamin laki laki agama budha warganegara indonesia alamat jl yupiter bs 1

=== case_010 ===
p u t u s a nnomor 699 pdt g 2024 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang mengadili perkara perdata telahmenjatuhkan putusan sebagai berikut dalam perkara gugatan antara henry najoan s h dalam hal ini bertindak untuk dan atas nama pt sativa dwi makmur beralamat di jalan h r muhammad no 161 surabaya berdasarkan salinanakta pernyataan keputusan rapat um

=== case_011 ===
halaman 1 putusan nomor 181 pdt g 2024 pn sby putusan nomor 181 pdt g 2024 pn sby demi keadilan berdasarkan ketuhanan yang maha esa pengadilan ne

In [71]:
# Cek distribusi dan penyebab Unknown
df_unknown = df_cases[df_cases['label_putusan'] == -1][
    ['case_id', 'word_count', 'amar_putusan', 'amar_lengkap']
]

print(f'Total Unknown: {len(df_unknown)}')
print()
print(df_unknown[['case_id', 'word_count', 'amar_putusan', 'amar_lengkap']].to_string(index=False))

Total Unknown: 0

Empty DataFrame
Columns: [case_id, word_count, amar_putusan, amar_lengkap]
Index: []


In [72]:
# Debug: cek case yang masih Unknown
df_unknown = df_cases[df_cases['label_putusan'] == -1][
    ['case_id', 'word_count', 'amar_putusan', 'amar_lengkap']
]

print(f'Total Unknown: {len(df_unknown)}')
print()
for _, row in df_unknown.iterrows():
    print(f"case_id      : {row['case_id']}")
    print(f"word_count   : {row['word_count']}")
    print(f"amar_putusan : {row['amar_putusan']}")
    print(f"amar_lengkap : {row['amar_lengkap'][:200] if row['amar_lengkap'] else 'KOSONG'}")
    print()

Total Unknown: 0



In [73]:
for _, row in df_cases[df_cases['label_putusan'] == -1].iterrows():
    # Buka file txt langsung
    path = f"data/raw/{row['case_id']}.txt"
    with open(path, 'r', encoding='utf-8') as f:
        teks = f.read()
    
    print(f"=== {row['case_id']} ({row['word_count']} kata) ===")
    print(f"amar_putusan metadata : '{row['amar_putusan']}'")
    print(f"amar_lengkap ekstrak  : '{row['amar_lengkap'][:300]}'")
    print()

In [74]:
for _, row in df_cases[df_cases['label_putusan'] == -1].iterrows():
    print(f"=== {row['case_id']} ({row['word_count']} kata) ===")
    print(f"amar_lengkap : '{row['amar_lengkap'][:300] if row['amar_lengkap'] else 'KOSONG'}'")
    print()

## Cell 6 — Validasi & Ringkasan Output

In [75]:
total        = len(df_cases)
ada_fakta    = (df_cases['ringkasan_fakta'].str.len() > 50).sum()
ada_argumen  = (df_cases['argumen_hukum'].str.len()   > 50).sum()
ada_amar     = (df_cases['amar_lengkap'].str.len()    > 20).sum()
label_kabul  = (df_cases['label_putusan'] == 1).sum()
label_tolak  = (df_cases['label_putusan'] == 0).sum()
label_unknwn = (df_cases['label_putusan'] == -1).sum()
avg_wc       = df_cases['word_count'].mean()

print('=' * 62)
print('  RINGKASAN TAHAP 2 — CASE REPRESENTATION')
print('=' * 62)
print(f'  Total kasus              : {total}')
print(f'  Jumlah kolom CSV         : {len(df_cases.columns)}')
print()
print('  Ekstraksi Konten:')
print(f'    Ringkasan fakta        : {ada_fakta}/{total}')
print(f'    Argumen hukum          : {ada_argumen}/{total}')
print(f'    Amar putusan lengkap   : {ada_amar}/{total}')
print()
print('  Distribusi Label Putusan:')
print(f'    Dikabulkan   (1)       : {label_kabul}')
print(f'    Ditolak/NO   (0)       : {label_tolak}')
print(f'    Unknown      (-1)      : {label_unknwn}')
print()
print(f'  Rata-rata word count     : {avg_wc:,.0f} kata/dokumen')
print()
print('  File output:')
print(f'    {PATH_CSV_OUT}')
print(f'    {PATH_JSON_OUT}')
print('=' * 62)
print()

# Preview sesuai contoh kolom di soal
cols_preview = ['case_id', 'no_perkara', 'tanggal', 'ringkasan_fakta', 'pasal', 'pihak', 'argumen_hukum', 'amar_lengkap', 'word_count', 'top_terms', 'label_putusan', 'text_full']
print('Preview 3 baris pertama (kolom sesuai soal):')
df_cases[cols_preview].head(3)

  RINGKASAN TAHAP 2 — CASE REPRESENTATION
  Total kasus              : 54
  Jumlah kolom CSV         : 16

  Ekstraksi Konten:
    Ringkasan fakta        : 54/54
    Argumen hukum          : 54/54
    Amar putusan lengkap   : 54/54

  Distribusi Label Putusan:
    Dikabulkan   (1)       : 30
    Ditolak/NO   (0)       : 24
    Unknown      (-1)      : 0

  Rata-rata word count     : 12,204 kata/dokumen

  File output:
    data/processed/cases.csv
    data/processed/cases.json

Preview 3 baris pertama (kolom sesuai soal):


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,argumen_hukum,amar_lengkap,word_count,top_terms,label_putusan,text_full
0,case_001,1124/Pdt.G/2024/PN Sby,21 Oktober 2024,hal 1 dari 50 putusan nomor 1124 pdt g 2024 pn sby putusan nomor 1124 pdt g 2024 pn sby demi keadilan berdasarkan ke...,"Pasal 1365, Pasal 1320, Pasal 1338, Pasal 1321, Pasal 1320 dan",Sukariyadi Rudi Meidiyanto vs. Fiona Magdalena Yapsawaky,nya menimbang bahwa maksud dan tujuan dari gugatan dari penggugat tersebut diatas menimbang bahwa gugatan penggugat ...,mengadili dalam konpensi dalam eksepsi menolak eksepsi dari tergugat dalam pokok perkara 1 mengabulkan gugatan pengg...,16920,"fotokopi, tergugat, penggugat, nomor, tanda, diberi, tanggal, rekonpensi, invoice, kapal",1,hal 1 dari 50 putusan nomor 1124 pdt g 2024 pn sby putusan nomor 1124 pdt g 2024 pn sby demi keadilan berdasarkan ke...
1,case_002,1225/Pdt.G/2024/PN Sby,18 Nopember 2024,menimbang bahwa penggugat dengan surat gugatan tanggal 11nopember 2024 yang diterima dan didaftarkan di kepaniteraan...,"Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332",Kandar Panjaitan vs. Johan Sugiharto Se,menimbang bahwa maksud dan tujuan gugatan penggugat yang padapokoknya adalah sebagaimana tersebut di atas dalam konp...,mengadili dalam konpensi dalam eksepsi menerima eksepsi dari tergugat dalam pokok perkara menyatakan gugatan penggug...,9165,"penggugat, tergugat, bukti, selanjutnya, fotokopi, diberi, tanda, gugatan, pekerjaan, bangunan",0,p u t u s a nnomor 1225 pdt g 2024 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya...
2,case_003,75/Pdt.G/2025/PN Sby,16 Januari 2025,menimbang bahwa penggugat dengan surat gugatan tanggal 13 januari 2025 yang diterima dan didaftarkan di kepaniteraan...,"Pasal 1320, Pasal 1243, Pasal 1338, Pasal 118 ayat 1, Pasal 123, Pasal 227 ayat 1, Pasal 180, Pasal 1267, Pasal 1238...",Pt Multi Bangun Indonesia vs. Pt Masa Sinar Mulia,menimbang bahwa maksud gugatan penggugat adalah sebagaimana tersebut diatas menimbang bahwa atas gugatan tersebut te...,mengadili 1 menyatakan bahwa tergugat yang telah dipanggil dengan patut untuk datang menghadap dipersidangan tidak h...,6515,"tergugat, penggugat, puluh, ratus, rupiah, tanggal, juta, lima, kepada, ribu",1,hal 1 dari 21 putusan nomor 75 pdt g 2025 pn sby putusan nomor 75 pdt g 2025 pn sby demi keadilan berdasarkan ketuha...


# Kumpulan Debug

In [76]:
# Cek 500 karakter pertama case_001 untuk lihat pola nama pihak
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

# Cari semua kemunculan kata 'penggugat' dan konteksnya
import re
for m in re.finditer(r'.{0,80}penggugat.{0,80}', teks, re.IGNORECASE):
    print(repr(m.group()))
    print()
    if m.start() > 3000:  # cukup 3000 karakter pertama saja
        break

'ngadilan negeri surabaya pada tanggal 15 oktober 2024 no 4718 hk x 2024 sebagai penggugat m e l a w a n fiona magdalena yapsawaky pekerjaan direktur cv bali marine servi'

' saksi yang diajukan dalam persidangan tentang duduknya perkara menimbang bahwa penggugat dalam surat gugatannya tertanggal 15 oktober 2024 yang terdaftar di kepaniteraa'

' 1124 pdt g 2024 pn sby telah mengajukan gugatan sebagai berikut 1 bahwa antara penggugat dan tergugat terdapat suatu hubungan hukum berdasarkan perjanjian kerjasama nom'

'bali 2 bahwa dalam pasal 2 dan pasal 3 perjanjian kerjasama tersebut dinyatakan penggugat menyerahkan kepada tergugat penggunaan atas ruangan souvenir dct benoa marina b'

'naan perjanjian ini maka apabila tidak diperoleh penyelesaian secara musyawarah penggugat dan tergugat memilih menyelesaikan perselisihan tersebut melalui pengadilan neg'



In [77]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

# Test gabung huruf berspasi
teks_gabung = re.sub(r'\b([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\b',
                     lambda m: ''.join(m.groups()), teks)

# Cek apakah 'melawan' berhasil digabung
print('Ada "melawan" setelah gabung:', 'melawan' in teks_gabung)
print()

# Cek 200 karakter sekitar 'melawan'
m = re.search(r'.{0,100}melawan.{0,100}', teks_gabung)
if m:
    print(repr(m.group()))

Ada "melawan" setelah gabung: True

' kantor tersebut tindakan ini tidak hanya mencederai perjanjian juga memenuhi kualifikasi perbuatan melawan hukum sebagaimana tergugat nanti uraikan pada bagian rekonvensi 4 bahwa adalah menjadi sangat wajar'


In [78]:
# Cari pola 'sebagai penggugat' dan sekitarnya
for m in re.finditer(r'.{0,150}sebagai penggugat.{0,150}', teks_gabung, re.IGNORECASE):
    print(repr(m.group()))
    print()

' kuasa khusus tanggal 12 agustus 2024 yang telah didaftar pada kepaniteraan pengadilan negeri surabaya pada tanggal 15 oktober 2024 no 4718 hk x 2024 sebagai penggugat melawa n fiona magdalena yapsawaky pekerjaan direktur cv bali marine service alamat jalan nuansa timur vii 7 tmn griya kel jimbaran kec kuta selatan '

'agian rekonpensi berikut ini untuk selanjutnya penggugat dalam konpensi disebut sebagai tergugat rekonpensi sedangkan tergugat dalam konpensi disebut sebagai penggugat rekonpensi 1 bahwa penggugat rekonpensi adalah direktur juga sekutu pada perusahaan berbentuk comanditer vernotschaap bernama cv bali marine service '



In [79]:
# Cari kata setelah 'melawan' yang bukan 'hukum'
for m in re.finditer(r'.{0,100}(?<!perbuatan )melawan(?!\s+hukum).{0,100}', teks_gabung, re.IGNORECASE):
    print(repr(m.group()))
    print()

'iknya tergugatlah yang meminta validitas dan akurasi tagihan yang dikemukakan oleh penggugat adalah melawan logika akal sehat manakala pihak yang menagih malah justru meminta bukti terhadap pihak yang ditagi'



In [80]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

teks_norm = re.sub(r'melawa\s+n\b', 'melawan', teks)

# Cek 1000 karakter pertama untuk lihat struktur awal dokumen
print(repr(teks_norm[:1000]))

'hal 1 dari 50 putusan nomor 1124 pdt g 2024 pn sby putusan nomor 1124 pdt g 2024 pn sby demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri surabaya yang memeriksa dan mengadili perkara perdata gugatan dalam peradilan tingkat pertama telah menjatuhkan putusan sebagai berikut dalam perkara antara sukariyadi rudi meidiyanto jabatan direktur utama pt pelindo properti indonesia berdasarkan akta notaris yahya abdullah waber sh nomor 5 tanggal 5 desember 2014 sebagaimana telah disahkan dengan keputusan menteri hukum dan ham nomor ahu 39434 40 10 2014 tanggal 12 desember 2014 tentang pengesahan pendirian badan hukum perseroan terbatas pt pelindo properti indonesia dan telah mengalami perubahan terakhir dinyatakan dalam akta pernyataan keputusan sirkuler pt pelindo properti indonesia nomor 23 tanggal 11 november 2022 yang dibuat di hadapan notaris illuminatia penny sh m kn sebagaimana telah diterima pemberitahuan perubahan data perseroan terbatas melalui surat keputusan menter

In [81]:
for case in ['case_002', 'case_003']:
    with open(f'data/raw/{case}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()
    
    teks_norm = re.sub(r'melawa\s+n\b', 'melawan', teks)
    print(f'=== {case} ===')
    print(repr(teks_norm[:800]))
    print()

=== case_002 ===
'p u t u s a nnomor 1225 pdt g 2024 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya yang memeriksa dan memutus perkaraperdata pada tingkat pertama telah menjatuhkan putusan sebagai berikutdalam perkara gugatan antara kandar panjaitan nik ktp 3578152107420002 jenis kelamin laki laki tempat tanggal lahir tapanuli utara 21 07 1942 umur83 tahun agama kristen warga negara indonesiapekerjaan purn tni al veteran alamat jln rembang no 95 surabaya perumahan g walk blok jj no 06 citraland surabaya dalam hal ini telah memberikan kuasa kepada erwinrudi agusman sibarani s h m h dan efendipanjaitan s h para advokat dan konsultanhukum berkantor di kantor hukum erwin sibarani sh mh efendi panjaitan sh rekan yangberalamat di jalan brigjen katamso no 222 224 bloka6 wedoro si'

=== case_003 ===
'hal 1 dari 21 putusan nomor 75 pdt g 2025 pn sby putusan nomor 75 pdt g 2025 pn sby demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri surabaya 

In [82]:
for case in ['case_001', 'case_002', 'case_003']:
    with open(f'data/raw/{case}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()
    
    teks_norm = re.sub(r'melawa\s+n\b', 'melawan', teks)
    teks_norm = re.sub(r'\b([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\b',
                       lambda m: ''.join(m.groups()), teks_norm)
    
    # Cari semua kemunculan 'melawan' dan konteksnya
    print(f'=== {case} ===')
    for m in re.finditer(r'.{0,50}melawan.{0,150}', teks_norm, re.IGNORECASE):
        print(repr(m.group()))
    print()

=== case_001 ===
'ai perjanjian juga memenuhi kualifikasi perbuatan melawan hukum sebagaimana tergugat nanti uraikan pada bagian rekonvensi 4 bahwa adalah menjadi sangat wajar manakala ruangan kantor yang semula diperjanjikan'
'si tagihan yang dikemukakan oleh penggugat adalah melawan logika akal sehat manakala pihak yang menagih malah justru meminta bukti terhadap pihak yang ditagih sebagaimana didalilkan oleh penggugat pada dalil'
' asas kepatutan tindakan mana merupakan perbuatan melawan hukum yang merugikan penggugat rekonpensi 18 bahwa upaya penggugat rekonpensi yang dalam melakukan surat menyurat maupun segala tindakannya yang menc'
'ekonpensi sepatutnya dinyatakan sebagai perbuatan melawan hukum yang dilakukan oleh tergugat rekonpensi 19 bahwa oleh karena segala tindakan yang dilakukan oleh tergugat rekonpensi dilakukan dengan memanfaat'
' rekonpensi telah nyata nyata melakukan perbuatan melawan hukum sebagaimana dimaksud dalam ketentuan pasal 1365 kuhperdata maka sudah sepatutn

In [83]:
for case in ['case_001', 'case_002', 'case_003']:
    with open(f'data/raw/{case}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()
    
    print(f'=== {case} ===')
    # Cari kata 'tergugat' di 1000 karakter pertama
    m = re.search(r'.{0,100}(?:lawan|versus|vs\.?)\s+([a-z].{0,200})', teks[:2000], re.IGNORECASE)
    if m:
        print('Ketemu lawan/vs:', repr(m.group()))
    else:
        # Cari nama setelah tanda pemisah di awal dokumen
        print('500-1500 char:')
        print(repr(teks[500:1500]))
    print()

=== case_001 ===
500-1500 char:
' dengan keputusan menteri hukum dan ham nomor ahu 39434 40 10 2014 tanggal 12 desember 2014 tentang pengesahan pendirian badan hukum perseroan terbatas pt pelindo properti indonesia dan telah mengalami perubahan terakhir dinyatakan dalam akta pernyataan keputusan sirkuler pt pelindo properti indonesia nomor 23 tanggal 11 november 2022 yang dibuat di hadapan notaris illuminatia penny sh m kn sebagaimana telah diterima pemberitahuan perubahan data perseroan terbatas melalui surat keputusan menteri hukum dan ham ri nomor ahu ah 01 09 0075488 tanggal 14 november 2022 perihal penerimaan pemberitahuan perubahan data perseroan pt pelindo properti indonesia alamat pelindo office place tower jalan perak barat no 379 perak utara pabean cantikan surabaya dalam hal ini memberikan kuasa kepada hotmaraja b nainggolan s h yohanis selle s h cathryna gabrielle djong s h dan vincent suriadinata s h m h para advokat yang beralamat di komplek ketapang indah blok b2 no 33 3

In [84]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

teks_norm = re.sub(r'melawa\s+n\b', 'lawan', teks)
teks_norm = re.sub(r'\b([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\s([a-z])\b',
                   lambda m: ''.join(m.groups()), teks_norm)

# Cek apakah 'melawa n' berhasil diganti
print('Ada "melawa n":', bool(re.search(r'melawa\s+n', teks)))
print('Ada "lawan" setelah norm:', bool(re.search(r'(?<!\bperbuatan )lawan(?!\s+hukum)(?!\s+logika)', teks_norm)))
print()

# Cari semua 'lawan' yang bukan 'perbuatan melawan hukum'
for m in re.finditer(r'.{0,80}(?<!\bperbuatan )lawan(?!\s+hukum)(?!\s+logika).{0,150}', teks_norm, re.IGNORECASE):
    print(repr(m.group()))
    print()

Ada "melawa n": False
Ada "lawan" setelah norm: True

'uk menyatakan putusan ini dapat dijalankan terlebih dahulu walaupun terdapat perlawanan berupa bantahan banding maupun kasasi uitvoerbaar bij vooraad berdasarkan segala uraian sebagaimana tersebut diatas maka tergugat penggugat rekonpe'

'antah melalui eksepsi dan pokok perkara bahkan pihak tergugat juga melakukan perlawanan dengan mengajukan gugatan balik atau rekonpensi terhadap pihak penggugat konpensi dalam eksepsi menimbang bahwa dalam eksepsi ini pihak tergugat pa'



In [85]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

# Cari 'tergugat' di 2000 karakter pertama
print(repr(teks[1000:2500]))

' hukum dan ham ri nomor ahu ah 01 09 0075488 tanggal 14 november 2022 perihal penerimaan pemberitahuan perubahan data perseroan pt pelindo properti indonesia alamat pelindo office place tower jalan perak barat no 379 perak utara pabean cantikan surabaya dalam hal ini memberikan kuasa kepada hotmaraja b nainggolan s h yohanis selle s h cathryna gabrielle djong s h dan vincent suriadinata s h m h para advokat yang beralamat di komplek ketapang indah blok b2 no 33 34 jalan kh zainul arifin jkarta barat berdasarkan surat kuasa khusus tanggal 12 agustus 2024 yang telah didaftar pada kepaniteraan pengadilan negeri surabaya pada tanggal 15 oktober 2024 no 4718 hk x 2024 sebagai penggugat m e l a w a n fiona magdalena yapsawaky pekerjaan direktur cv bali marine service alamat jalan nuansa timur vii 7 tmn griya kel jimbaran kec kuta selatan kab badung bali sebagai tergugat pengadilan negeri tersebut\n\nhal 2 dari 50 putusan nomor 1124 pdt g 2024 pn sby telah membaca berkas perkara dan surat su

In [86]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                   lambda m: m.group().replace(' ', ''), teks)

# Cek apakah 'm e l a w a n' berhasil digabung
print('Ada "m e l a w a n" sebelum:', bool(re.search(r'm e l a w a n', teks)))
print('Ada "melawan" setelah norm:', bool(re.search(r'\bmelawan\b', teks_norm)))
print()

# Lihat sekitar kata melawan/lawan
for m in re.finditer(r'.{0,50}(?:melawan|lawan).{0,150}', teks_norm[:3000], re.IGNORECASE):
    print(repr(m.group()))
    print()

Ada "m e l a w a n" sebelum: True
Ada "melawan" setelah norm: True

' oktober 2024 no 4718 hk x 2024 sebagai penggugat melawan fiona magdalena yapsawaky pekerjaan direktur cv bali marine service alamat jalan nuansa timur vii 7 tmn griya kel jimbaran kec kuta selatan kab badun'



In [87]:
with open('data/raw/case_001.txt', 'r', encoding='utf-8') as f:
    teks = f.read()

teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                   lambda m: m.group().replace(' ', ''), teks)

# Test regex tergugat langsung
m = re.search(
    r'\b(?:me)?lawan\s+(?!hukum|logika|bantahan|perlawanan)([a-z][a-z\s\.]{2,60}?)(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|\s+bertindak|\s+yang\s+berkedudukan|\s+sebagai\s+tergugat|\s+tempat|\s+umur|\s+jenis)',
    teks_norm, re.IGNORECASE
)
print('Match:', m)
if m:
    print('Group 1:', repr(m.group(1)))

Match: <re.Match object; span=(1687, 1730), match='melawan fiona magdalena yapsawaky pekerjaan'>
Group 1: 'fiona magdalena yapsawaky'


In [88]:
print(ekstrak_pihak_dari_teks(teks))

Sukariyadi Rudi Meidiyanto vs. Fiona Magdalena Yapsawaky


In [89]:
import os
print(os.path.exists('data/processed/cases.csv'))
print(os.path.exists('data/processed/cases.json'))

True
True


In [90]:
with open('data/raw/case_039.txt', 'r', encoding='utf-8') as f:
    teks_039 = f.read()

print("Hasil:", ekstrak_pihak_dari_teks(teks_039))
print()
print("Berapa case yang masih Tidak Tertera setelah perbaikan:")
sisa = df_cases[df_cases['pihak'].str.contains('Tidak Tertera')]
print(f"{len(sisa)} dari {len(df_cases)}")

Hasil: Santi Boediyanto vs. 1 Go Gorry Lisayati

Berapa case yang masih Tidak Tertera setelah perbaikan:
0 dari 54


In [91]:
sisa_tidak_tertera = df_cases[df_cases['pihak'].str.contains('Tidak Tertera')]
print(f"Sisa dokumen dengan pihak Tidak Tertera: {len(sisa_tidak_tertera)} dari {len(df_cases)}")
print(sisa_tidak_tertera[['case_id', 'pihak']])

Sisa dokumen dengan pihak Tidak Tertera: 0 dari 54
Empty DataFrame
Columns: [case_id, pihak]
Index: []


In [92]:
import inspect
print(inspect.getsource(ekstrak_pihak_dari_teks))

def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di tengah kalimat akibat page-break PDF
    teks_norm = re.sub(
        r'halaman\s*\d+\s*(?:dari\s*\d+\s*)?putusan\s*no\.?\s*(?:mor)?\s*\d+\s*pdt\s*g\s*\d{4}\s*pn\s*sby',
        ' ', teks_norm, flags=re.IGNORECASE
    )

    nama_bulan = r'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'

    pemutus = (
        r'(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|'
        r'\s+dalam\s+kapasitas|\s+bertindak|\s+berdasarkan|\s+yang\s*berkedudukan|\s+berkedudukan|'
        r'\s+diwakili|\s+lawa

In [93]:
import inspect
print(inspect.getsource(ekstrak_pihak_dari_teks))
print()
print("=== TEST LANGSUNG case_005 ===")
with open('data/raw/case_005.txt', 'r', encoding='utf-8') as f:
    teks_005 = f.read()
print(ekstrak_pihak_dari_teks(teks_005))

def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di tengah kalimat akibat page-break PDF
    teks_norm = re.sub(
        r'halaman\s*\d+\s*(?:dari\s*\d+\s*)?putusan\s*no\.?\s*(?:mor)?\s*\d+\s*pdt\s*g\s*\d{4}\s*pn\s*sby',
        ' ', teks_norm, flags=re.IGNORECASE
    )

    nama_bulan = r'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'

    pemutus = (
        r'(?:\s+nik|\s+nip|\s+jabatan|\s+pekerjaan|\s+beralamat|\s+alamat|\s+dalam\s+hal|'
        r'\s+dalam\s+kapasitas|\s+bertindak|\s+berdasarkan|\s+yang\s*berkedudukan|\s+berkedudukan|'
        r'\s+diwakili|\s+lawa

In [94]:
df_bermasalah = df_cases[df_cases['pihak'].str.contains('Tidak Tertera', na=False)]
print(f'Total: {len(df_bermasalah)} dari {len(df_cases)}')
print(df_bermasalah[['case_id', 'pihak']].to_string(index=False))

Total: 0 dari 54
Empty DataFrame
Columns: [case_id, pihak]
Index: []


In [95]:
import inspect
print("=== gabung_huruf_berspasi ===")
print(inspect.getsource(gabung_huruf_berspasi))
print()
print("=== ekstrak_pihak_dari_teks ===")
print(inspect.getsource(ekstrak_pihak_dari_teks))
print()
print("=== TEST LANGSUNG case_009 ===")
with open('data/raw/case_009.txt', 'r', encoding='utf-8') as f:
    teks_009 = f.read()
print(ekstrak_pihak_dari_teks(teks_009))

=== gabung_huruf_berspasi ===
def gabung_huruf_berspasi(teks: str) -> str:
    teks = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                  lambda m: m.group().replace(' ', ''), teks)

    # PERBAIKAN: tangani 'm e l a w a n' yang menempel langsung ke kata
    # berikutnya tanpa spasi (mis. '...a nwilliam'), sehingga huruf 'n'
    # terakhir gagal ikut tergabung oleh pola umum di atas.
    teks = re.sub(r'\bmelawa\s*n(?=[a-z])', 'melawan ', teks)
    teks = re.sub(r'\blawa\s*n(?=[a-z])', 'lawan ', teks)

    return teks


=== ekstrak_pihak_dari_teks ===
def ekstrak_pihak_dari_teks(teks: str) -> str:
    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)

    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    # PERBAIKAN BARU: hapus watermark "halaman X [dari Y] putusan no Z pdt g
    # TAHUN pn sby" yang kadang menyelip di ten

In [96]:
sisa_tidak_tertera = df_cases[df_cases['pihak'].str.contains('Tidak Tertera')]
print(f"Sisa dokumen dengan pihak Tidak Tertera: {len(sisa_tidak_tertera)} dari {len(df_cases)}")
print(sisa_tidak_tertera[['case_id', 'pihak']])

Sisa dokumen dengan pihak Tidak Tertera: 0 dari 54
Empty DataFrame
Columns: [case_id, pihak]
Index: []


In [97]:
import re

def cek_kualitas_pihak(pihak_str):
    """Tandai kemungkinan masalah pada hasil ekstraksi pihak."""
    masalah = []

    if 'tidak tertera' in pihak_str.lower():
        masalah.append('ADA YANG KOSONG')

    # Kata-kata yang mengindikasikan capture "menelan" terlalu jauh
    kata_sampah = ['yang', 'dan', 'halaman', 'dari', 'putusan no', 'di jl', 'bertempat']
    for kw in kata_sampah:
        if kw in pihak_str.lower():
            masalah.append(f"mengandung '{kw}'")

    # Nama yang terlalu panjang (>5 kata per pihak) kemungkinan ikut menelan teks lain
    for sisi in pihak_str.split(' vs. '):
        jumlah_kata = len(sisi.split())
        if jumlah_kata > 5:
            masalah.append(f"sisi '{sisi[:30]}...' terlalu panjang ({jumlah_kata} kata)")

    return '; '.join(masalah) if masalah else 'OK'

df_cases['cek_pihak'] = df_cases['pihak'].apply(cek_kualitas_pihak)

# Tampilkan hanya yang BERMASALAH (bukan 'OK')
bermasalah = df_cases[df_cases['cek_pihak'] != 'OK']
print(f"Total kemungkinan bermasalah: {len(bermasalah)} dari {len(df_cases)}")
print()
pd.set_option('display.max_colwidth', 120)
print(bermasalah[['case_id', 'pihak', 'cek_pihak']].to_string(index=False))

Total kemungkinan bermasalah: 7 dari 54

 case_id                                                                pihak                                                                          cek_pihak
case_006                           Daniel Douglas Wijaya vs. 1 Raymond Wijaya                                                                   mengandung 'dan'
case_007     Effendi Surjonogo vs. 1 Pt Tokio Marine Life Insurance Indonesia                  sisi '1 Pt Tokio Marine Life Insuran...' terlalu panjang (7 kata)
case_024                 Rizka Kusuma Wardani A Md Keb vs. Rusilowati Handojo mengandung 'dan'; sisi 'Rizka Kusuma Wardani A Md Keb...' terlalu panjang (6 kata)
case_033                  1 Yessy Norawati Mardani Se vs. Pt Diparanu Rucitra                                                                   mengandung 'dan'
case_043 Lim Giok Hua vs. Pt Suryainti Permata Tbk Perseroan Terbatas Terbuka                  sisi 'Pt Suryainti Permata Tbk Perse...' terlalu panjang (7

In [98]:
import re

def cek_kualitas_pihak(pihak_str):
    """
    Tandai kemungkinan masalah pada hasil ekstraksi pihak.
    Hanya menandai indikator yang benar-benar menunjukkan kegagalan
    ekstraksi (capture menelan kata sampah), bukan nama PT/gelar yang
    secara alami panjang.
    """
    masalah = []

    if 'tidak tertera' in pihak_str.lower():
        masalah.append('ADA YANG KOSONG')

    # Kata utuh (dicek dengan word boundary \b) yang menandakan capture
    # salah -- BUKAN sekadar substring kebetulan dalam sebuah nama
    kata_sampah = [
        r'\byang\b', r'\bhalaman\b', r'\bdari\b', r'\bputusan\s*no\b',
        r'\bbertempat\b', r'\bberkedudukan\b', r'\bdalam\s+hal\b',
        r'\bsebagai\s+tergugat\b', r'\bsebagai\s+penggugat\b',
    ]
    for pat in kata_sampah:
        if re.search(pat, pihak_str, re.IGNORECASE):
            label = pat.replace(r'\b', '').replace(r'\s+', ' ').replace(r'\s*', ' ')
            masalah.append(f"mengandung '{label}'")

    # Ambang panjang nama dinaikkan ke 9 kata (nama PT + status hukum
    # lengkap, atau nama + beberapa gelar, wajar sampai ~8-9 kata)
    AMBANG_KATA = 9
    for sisi in pihak_str.split(' vs. '):
        jumlah_kata = len(sisi.split())
        if jumlah_kata > AMBANG_KATA:
            masalah.append(f"sisi '{sisi[:30]}...' terlalu panjang ({jumlah_kata} kata)")

    return '; '.join(masalah) if masalah else 'OK'


df_cases['cek_pihak'] = df_cases['pihak'].apply(cek_kualitas_pihak)

bermasalah = df_cases[df_cases['cek_pihak'] != 'OK']
print(f"Total kemungkinan bermasalah: {len(bermasalah)} dari {len(df_cases)}")
print()
import pandas as pd
pd.set_option('display.max_colwidth', 120)
print(bermasalah[['case_id', 'pihak', 'cek_pihak']].to_string(index=False))

Total kemungkinan bermasalah: 0 dari 54

Empty DataFrame
Columns: [case_id, pihak, cek_pihak]
Index: []


In [99]:
for cid in ['case_033', 'case_041']:
    with open(f'data/raw/{cid}.txt', 'r', encoding='utf-8') as f:
        teks = f.read()

    teks_norm = re.sub(r'\b([a-z])(\s[a-z]){2,9}\b',
                       lambda m: m.group().replace(' ', ''), teks)
    teks_norm = re.sub(r'\bmelawa\s*n(?=[a-z0-9])', 'melawan ', teks_norm)
    teks_norm = re.sub(r'\blawa\s*n(?=[a-z0-9])', 'lawan ', teks_norm)

    idx = teks_norm.find('lawan')
    print(f"=== {cid} ===")
    print(teks_norm[max(0,idx-10):idx+200])
    print()

=== case_033 ===
nggugat melawan halaman 1 putusan nomor 1277 pdt g 2024 pn sby

pt diparanu rucitra beralamat domisili di jalan kertomenanggal 9nomor 11 kelurahan dukuh menanggal kecamatangayungan kota surabaya diwakili oleh y

=== case_041 ===
penggugat lawan

halaman 2 dari 40 putusan no 237 pdt g 2025 pn sby 1 williem fredrejk m surabaya 16 mei 1985 alamat di manukan lor 8c 20 kota surabaya sebagai tergugat i 2 tan jeffry setiawan sidoarjo 11 mei 1



# Code apabila masih ada file yang nyangkut

In [100]:
import os, re

def gabung_huruf_berspasi(teks: str) -> str:
    """Rapikan huruf yang terpisah spasi akibat tipografi PDF, mis. 'p u t u s a n' -> 'putusan'."""
    pola = re.compile(r'\b(?:[a-zA-Z]\s{1,2}){2,}[a-zA-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)


def cek_jenis_dokumen(case_id: str, dir_txt: str = 'data/raw') -> str:
    """
    Deteksi apakah dokumen adalah PUTUSAN substantif atau PENETAPAN
    (administratif/sela/perdamaian), berdasarkan frasa pembuka resmi dokumen.

    Prioritas: 'menjatuhkan putusan' dicek LEBIH DULU, karena banyak Putusan
    asli juga menyebut 'penetapan hari sidang' sebagai riwayat administratif
    persidangan -- itu bukan tanda dokumen tersebut adalah Penetapan.
    """
    path = os.path.join(dir_txt, f'{case_id}.txt')
    with open(path, 'r', encoding='utf-8') as f:
        teks_awal = f.read()[:800].lower()

    teks_rapi = gabung_huruf_berspasi(teks_awal)

    pola_putusan = re.compile(r'telah\s*menjatuhkan\s*putusan|menjatuhkan\s*putusan')
    pola_penetapan_jelas = re.compile(
        r'telah\s*menjatuhkan\s*penetapan|telah\s*memberikan\s*penetapan|memberikan\s*penetapan'
    )
    pola_penetapan_sela = re.compile(r'penetapan\s*hari\s*sidang|penetapan\s*sela')

    if pola_putusan.search(teks_rapi):
        return 'PUTUSAN'
    elif pola_penetapan_jelas.search(teks_rapi) or pola_penetapan_sela.search(teks_rapi):
        return 'PENETAPAN'
    return 'TIDAK_TERDETEKSI'


df_cases['jenis_dokumen'] = df_cases['case_id'].apply(cek_jenis_dokumen)
print(df_cases['jenis_dokumen'].value_counts())
print()
print(df_cases[df_cases['jenis_dokumen'] != 'PUTUSAN'][['case_id', 'word_count', 'jenis_dokumen']])

jenis_dokumen
PUTUSAN    54
Name: count, dtype: int64

Empty DataFrame
Columns: [case_id, word_count, jenis_dokumen]
Index: []


In [101]:
# ── Buang dokumen yang bukan Putusan substantif ────────────────────────────
sebelum = len(df_cases)
df_cases_final = df_cases[df_cases['jenis_dokumen'] == 'PUTUSAN'].reset_index(drop=True)
sesudah = len(df_cases_final)

print('=' * 62)
print('  VALIDASI CASE BASE SETELAH FILTER JENIS DOKUMEN')
print('=' * 62)
print(f'  Dokumen sebelum filter      : {sebelum}')
print(f'  Dokumen setelah filter      : {sesudah}')
print(f'  Dibuang (Penetapan/lainnya) : {sebelum - sesudah}')
print()

if sesudah >= 30:
    print(f'  STATUS: ✅ AMAN — {sesudah} dokumen masih memenuhi syarat minimum tugas (≥ 30)')
    print('           Tidak perlu scraping ulang di Tahap 1.')
else:
    print(f'  STATUS: ⚠️  KURANG — {sesudah} dokumen di bawah syarat minimum (30)')
    print('           Perlu menambah dokumen baru lewat scraping Tahap 1.')

# ── Simpan ulang sebagai cases.csv & cases.json FINAL ──────────────────────
df_cases_final = df_cases_final.drop(columns=['jenis_dokumen'])  # kolom bantu, tak perlu disimpan
df_cases_final.to_csv(PATH_CSV_OUT, index=False, encoding='utf-8-sig')
print(f'\n✅ CSV final tersimpan  -> {PATH_CSV_OUT} ({len(df_cases_final)} baris)')

import json
records = df_cases_final.drop(columns=['text_full']).to_dict(orient='records')
with open(PATH_JSON_OUT, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f'✅ JSON final tersimpan -> {PATH_JSON_OUT} (tanpa kolom text_full)')

df_cases = df_cases_final  # pastikan variabel df_cases dipakai konsisten ke tahap berikutnya

  VALIDASI CASE BASE SETELAH FILTER JENIS DOKUMEN
  Dokumen sebelum filter      : 54
  Dokumen setelah filter      : 54
  Dibuang (Penetapan/lainnya) : 0

  STATUS: ✅ AMAN — 54 dokumen masih memenuhi syarat minimum tugas (≥ 30)
           Tidak perlu scraping ulang di Tahap 1.

✅ CSV final tersimpan  -> data/processed/cases.csv (54 baris)
✅ JSON final tersimpan -> data/processed/cases.json (tanpa kolom text_full)


In [102]:
# ═══════════════════════════════════════════════════════════════════════
# Cari 4 case_id yang dibuang Tahap 2 (selisih antara sebelum & sesudah filter)
# Jalankan ini SETELAH cell filter "58 -> 54" tadi, di notebook Tahap 2
# ═══════════════════════════════════════════════════════════════════════
import os

# Daftar case_id yang ADA di folder mentah (58 file)
DIR_PDF = 'data/raw/pdf'
case_id_mentah = set(f.replace('.pdf', '') for f in os.listdir(DIR_PDF) if f.endswith('.pdf'))

# Daftar case_id yang LOLOS dan masuk cases.csv final (54 baris)
case_id_lolos = set(df_cases['case_id'])  # FIX: pakai df_cases (variabel final), bukan df

# Selisihnya = yang dibuang
case_id_terbuang = sorted(case_id_mentah - case_id_lolos)

print(f'Total file mentah   : {len(case_id_mentah)}')
print(f'Total lolos filter  : {len(case_id_lolos)}')
print(f'Total terbuang      : {len(case_id_terbuang)}')
print()
print('case_id yang dibuang Tahap 2:')
for cid in case_id_terbuang:
    print(f'  {cid}')


Total file mentah   : 54
Total lolos filter  : 54
Total terbuang      : 0

case_id yang dibuang Tahap 2:


In [103]:
# ═══════════════════════════════════════════════════════════════════════
# Versi robust: baca langsung dari cases.csv (tidak bergantung nama variabel)
# Lalu HAPUS file fisik 4 case yang terbuang, supaya folder rapi.
# Jalankan ini KAPAN SAJA setelah cases.csv final (54 baris) sudah tersimpan.
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd

DIR_PDF = 'data/raw/pdf'

# 1) Cari case_id yang terbuang
case_id_mentah = set(f.replace('.pdf', '') for f in os.listdir(DIR_PDF) if f.endswith('.pdf'))
df_final = pd.read_csv('data/processed/cases.csv', encoding='utf-8-sig')
case_id_lolos = set(df_final['case_id'])

case_id_terbuang = sorted(case_id_mentah - case_id_lolos)

print(f'Total file mentah  : {len(case_id_mentah)}')
print(f'Total lolos filter : {len(case_id_lolos)}')
print(f'Total terbuang     : {len(case_id_terbuang)}')
print()
print('case_id yang akan DIHAPUS dari disk:')
for cid in case_id_terbuang:
    print(f'  {cid}')

if len(case_id_terbuang) == 0:
    print('\n✅ Tidak ada yang perlu dihapus, sudah rapi.')
else:
    print()
    konfirmasi = input(f'Yakin hapus {len(case_id_terbuang)} file di atas? (ketik "ya" untuk lanjut): ')
    if konfirmasi.strip().lower() == 'ya':
        for case_id in case_id_terbuang:
            for path in [
                f'data/raw/pdf/{case_id}.pdf',
                f'data/raw/{case_id}.txt',
                f'data/processed/tokens/{case_id}_tokens.json',
            ]:
                if os.path.exists(path):
                    os.remove(path)
                    print(f'  🗑️  {path}')

        # Bersihkan juga metadata_raw.csv & preprocessing_summary.csv biar konsisten
        for path_meta in ['data/processed/metadata_raw.csv', 'logs/preprocessing_summary.csv']:
            if os.path.exists(path_meta):
                dfm = pd.read_csv(path_meta)
                sebelum = len(dfm)
                dfm = dfm[~dfm['case_id'].isin(case_id_terbuang)]
                dfm.to_csv(path_meta, index=False, encoding='utf-8-sig')
                print(f'✅ {path_meta} dibersihkan: {sebelum} -> {len(dfm)} baris')

        sisa = len([f for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])
        print(f'\nSelesai. Sisa file fisik di folder: {sisa} (harus sama dengan {len(case_id_lolos)} di cases.csv)')
    else:
        print('Dibatalkan, tidak ada file yang dihapus.')


Total file mentah  : 54
Total lolos filter : 54
Total terbuang     : 0

case_id yang akan DIHAPUS dari disk:

✅ Tidak ada yang perlu dihapus, sudah rapi.


In [104]:
# ═══════════════════════════════════════════════════════════════════════
# Sinkronkan file fisik (PDF, TXT, token JSON) dengan cases.csv FINAL
# Menghapus file-file yang case_id-nya TIDAK ada di cases.csv (54 baris),
# supaya Tahap 3 dst hanya melihat dokumen yang benar-benar valid.
#
# CATATAN: versi ini TIDAK pakai input() interaktif, supaya aman dijalankan
# lewat "Run All" tanpa macet menunggu input. Ganti KONFIRMASI_HAPUS di
# bawah jadi False dulu kalau cuma mau LIHAT daftarnya tanpa menghapus.
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd

KONFIRMASI_HAPUS = True   # set False untuk mode pratinjau (dry-run, tidak menghapus apa pun)

DIR_PDF = 'data/raw/pdf'
DIR_TXT = 'data/raw'
DIR_TOKENS = 'data/processed/tokens'

# 1) Daftar case_id yang ADA secara fisik (dari file PDF)
case_id_mentah = set(f.replace('.pdf', '') for f in os.listdir(DIR_PDF) if f.endswith('.pdf'))

# 2) Daftar case_id yang LOLOS dan ada di cases.csv FINAL
df_final = pd.read_csv('data/processed/cases.csv', encoding='utf-8-sig')
case_id_lolos = set(df_final['case_id'])

# 3) Selisihnya = yang harus dihapus dari disk
case_id_terbuang = sorted(case_id_mentah - case_id_lolos)

print('=' * 62)
print('  SINKRONISASI FILE FISIK <-> cases.csv FINAL')
print('=' * 62)
print(f'  Total file mentah (PDF)     : {len(case_id_mentah)}')
print(f'  Total lolos di cases.csv    : {len(case_id_lolos)}')
print(f'  Total akan dihapus          : {len(case_id_terbuang)}')
print()

if len(case_id_terbuang) == 0:
    print('✅ Sudah sinkron, tidak ada file yang perlu dihapus.')
else:
    print('case_id yang akan dihapus dari disk:')
    for cid in case_id_terbuang:
        print(f'  - {cid}')
    print()

    if not KONFIRMASI_HAPUS:
        print('⚠️  MODE PRATINJAU (KONFIRMASI_HAPUS = False) — tidak ada file yang dihapus.')
        print('   Set KONFIRMASI_HAPUS = True lalu jalankan ulang cell ini untuk benar-benar menghapus.')
    else:
        dihapus = 0
        for case_id in case_id_terbuang:
            for path in [
                f'{DIR_PDF}/{case_id}.pdf',
                f'{DIR_TXT}/{case_id}.txt',
                f'{DIR_TOKENS}/{case_id}_tokens.json',
            ]:
                if os.path.exists(path):
                    os.remove(path)
                    dihapus += 1
                    print(f'  🗑️  dihapus: {path}')

        # Sinkronkan juga metadata_raw.csv & preprocessing_summary.csv (kalau ada)
        for path_meta in ['data/processed/metadata_raw.csv', 'logs/preprocessing_summary.csv']:
            if os.path.exists(path_meta):
                dfm = pd.read_csv(path_meta, encoding='utf-8-sig')
                sebelum = len(dfm)
                dfm = dfm[~dfm['case_id'].isin(case_id_terbuang)]
                dfm.to_csv(path_meta, index=False, encoding='utf-8-sig')
                print(f'  ✅ {path_meta} disinkronkan: {sebelum} -> {len(dfm)} baris')

        sisa = len([f for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])
        print()
        print(f'🎯 SELESAI. Total {dihapus} file fisik dihapus.')
        print(f'   Sisa file di data/raw/pdf/: {sisa} (harus = {len(case_id_lolos)} baris di cases.csv)')
        print(f'   Status: {"✅ SINKRON" if sisa == len(case_id_lolos) else "⚠️ MASIH BEDA, cek manual"}')


  SINKRONISASI FILE FISIK <-> cases.csv FINAL
  Total file mentah (PDF)     : 54
  Total lolos di cases.csv    : 54
  Total akan dihapus          : 0

✅ Sudah sinkron, tidak ada file yang perlu dihapus.


In [105]:
# ═══════════════════════════════════════════════════════════════════════
# Rename case_id agar berurutan kembali (case_001..case_054)
# Jalankan SETELAH file fisik disinkronkan (PDF/TXT/token sudah 54, sama
# dengan cases.csv). Cell ini merapikan SEMUA 6 lokasi sekaligus:
#   1. PDF di data/raw/pdf/
#   2. TXT di data/raw/
#   3. Token JSON di data/processed/tokens/ (jika ada)
#   4. metadata_raw.csv
#   5. cases.csv       <- hasil akhir Tahap 2, dipakai Tahap 3/4/5
#   6. cases.json       <- hasil akhir Tahap 2, dipakai Tahap 3/4/5
# ═══════════════════════════════════════════════════════════════════════
import os
import json
import pandas as pd

PATH_CASES_CSV  = 'data/processed/cases.csv'
PATH_CASES_JSON = 'data/processed/cases.json'
PATH_META_RAW   = 'data/processed/metadata_raw.csv'

# 1. Ambil daftar case_id final dari cases.csv, urutkan sesuai urutan numerik lama
df_cases = pd.read_csv(PATH_CASES_CSV, encoding='utf-8-sig')
df_cases = df_cases.sort_values('case_id').reset_index(drop=True)

# 2. Buat mapping: nama lama -> nama baru yang berurutan
mapping = {}
for idx, old_id in enumerate(df_cases['case_id'], start=1):
    new_id = f'case_{idx:03d}'
    mapping[old_id] = new_id

print(f'Total case yang akan di-rename: {len(mapping)}')
print('Contoh mapping (5 pertama):')
for old, new in list(mapping.items())[:5]:
    print(f'  {old}  ->  {new}')
print('...')
print()

# 3. Rename file PDF (lewat nama sementara agar tidak tabrakan)
renamed_pdf = 0
for old_id, new_id in mapping.items():
    src = f'data/raw/pdf/{old_id}.pdf'
    if os.path.exists(src):
        os.rename(src, f'data/raw/pdf/__tmp__{new_id}.pdf')
        renamed_pdf += 1
for old_id, new_id in mapping.items():
    tmp = f'data/raw/pdf/__tmp__{new_id}.pdf'
    if os.path.exists(tmp):
        os.rename(tmp, f'data/raw/pdf/{new_id}.pdf')
print(f'✅ PDF di-rename: {renamed_pdf} file')

# 4. Rename file TXT mentah
renamed_txt = 0
for old_id, new_id in mapping.items():
    src = f'data/raw/{old_id}.txt'
    if os.path.exists(src):
        os.rename(src, f'data/raw/__tmp__{new_id}.txt')
        renamed_txt += 1
for old_id, new_id in mapping.items():
    tmp = f'data/raw/__tmp__{new_id}.txt'
    if os.path.exists(tmp):
        os.rename(tmp, f'data/raw/{new_id}.txt')
print(f'✅ TXT di-rename: {renamed_txt} file')

# 5. Rename file token JSON, kalau ada
renamed_tok = 0
dir_tokens = 'data/processed/tokens'
if os.path.exists(dir_tokens):
    for old_id, new_id in mapping.items():
        src = f'{dir_tokens}/{old_id}_tokens.json'
        if os.path.exists(src):
            os.rename(src, f'{dir_tokens}/__tmp__{new_id}_tokens.json')
            renamed_tok += 1
    for old_id, new_id in mapping.items():
        tmp = f'{dir_tokens}/__tmp__{new_id}_tokens.json'
        if os.path.exists(tmp):
            os.rename(tmp, f'{dir_tokens}/{new_id}_tokens.json')
print(f'✅ Token JSON di-rename: {renamed_tok} file')

# 6. Update cases.csv (kolom case_id)
df_cases['case_id'] = df_cases['case_id'].map(mapping)
df_cases = df_cases.sort_values('case_id').reset_index(drop=True)
df_cases.to_csv(PATH_CASES_CSV, index=False, encoding='utf-8-sig')
print(f'✅ cases.csv diperbarui: {df_cases["case_id"].iloc[0]} .. {df_cases["case_id"].iloc[-1]}')

# 7. Update cases.json (kolom case_id) jika ada
if os.path.exists(PATH_CASES_JSON):
    with open(PATH_CASES_JSON, 'r', encoding='utf-8') as f:
        records = json.load(f)
    for rec in records:
        if rec.get('case_id') in mapping:
            rec['case_id'] = mapping[rec['case_id']]
    records = sorted(records, key=lambda r: r['case_id'])
    with open(PATH_CASES_JSON, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print('✅ cases.json diperbarui')

# 8. Update metadata_raw.csv jika ada
if os.path.exists(PATH_META_RAW):
    dfm = pd.read_csv(PATH_META_RAW, encoding='utf-8-sig')
    dfm = dfm[dfm['case_id'].isin(mapping.keys())]  # hanya yang masih lolos
    dfm['case_id'] = dfm['case_id'].map(mapping)
    dfm = dfm.sort_values('case_id').reset_index(drop=True)
    dfm.to_csv(PATH_META_RAW, index=False, encoding='utf-8-sig')
    print(f'✅ metadata_raw.csv diperbarui: {len(dfm)} baris')

# 9. Update logs/preprocessing_summary.csv jika ada
path_summary = 'logs/preprocessing_summary.csv'
if os.path.exists(path_summary):
    df_sum = pd.read_csv(path_summary)
    df_sum = df_sum[df_sum['case_id'].isin(mapping.keys())]
    df_sum['case_id'] = df_sum['case_id'].map(mapping)
    df_sum.to_csv(path_summary, index=False)
    print('✅ logs/preprocessing_summary.csv diperbarui')

# 10. Simpan mapping lama->baru sebagai log audit
os.makedirs('logs', exist_ok=True)
with open('logs/rename_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print('✅ Mapping lama->baru disimpan ke logs/rename_mapping.json')

# 11. Verifikasi akhir: jumlah file harus sama persis dengan baris CSV
n_pdf = len([f for f in os.listdir('data/raw/pdf') if f.endswith('.pdf')])
n_txt = len([f for f in os.listdir('data/raw') if f.endswith('.txt')])
n_csv = len(df_cases)

print()
print('=' * 62)
print(f'SELESAI. Case base sekarang berurutan: case_001 .. case_{len(mapping):03d}')
print(f'  PDF : {n_pdf}')
print(f'  TXT : {n_txt}')
print(f'  CSV : {n_csv}')
print(f'  Status: {"✅ SEMUA SINKRON" if n_pdf == n_txt == n_csv else "⚠️ MASIH BEDA, cek manual"}')
print('=' * 62)


Total case yang akan di-rename: 54
Contoh mapping (5 pertama):
  case_001  ->  case_001
  case_002  ->  case_002
  case_003  ->  case_003
  case_004  ->  case_004
  case_005  ->  case_005
...

✅ PDF di-rename: 54 file
✅ TXT di-rename: 54 file
✅ Token JSON di-rename: 54 file
✅ cases.csv diperbarui: case_001 .. case_054
✅ cases.json diperbarui
✅ metadata_raw.csv diperbarui: 54 baris
✅ logs/preprocessing_summary.csv diperbarui
✅ Mapping lama->baru disimpan ke logs/rename_mapping.json

SELESAI. Case base sekarang berurutan: case_001 .. case_054
  PDF : 54
  TXT : 54
  CSV : 54
  Status: ✅ SEMUA SINKRON


## Validasi & Ringkasan Output Setelah Pembersihan Dari File yang Masih Tidak sesuai

In [106]:
total        = len(df_cases)
ada_fakta    = (df_cases['ringkasan_fakta'].str.len() > 50).sum()
ada_argumen  = (df_cases['argumen_hukum'].str.len()   > 50).sum()
ada_amar     = (df_cases['amar_lengkap'].str.len()    > 20).sum()
label_kabul  = (df_cases['label_putusan'] == 1).sum()
label_tolak  = (df_cases['label_putusan'] == 0).sum()
label_unknwn = (df_cases['label_putusan'] == -1).sum()
avg_wc       = df_cases['word_count'].mean()

print('=' * 62)
print('  RINGKASAN TAHAP 2 — CASE REPRESENTATION')
print('=' * 62)
print(f'  Total kasus              : {total}')
print(f'  Jumlah kolom CSV         : {len(df_cases.columns)}')
print()
print('  Ekstraksi Konten:')
print(f'    Ringkasan fakta        : {ada_fakta}/{total}')
print(f'    Argumen hukum          : {ada_argumen}/{total}')
print(f'    Amar putusan lengkap   : {ada_amar}/{total}')
print()
print('  Distribusi Label Putusan:')
print(f'    Dikabulkan   (1)       : {label_kabul}')
print(f'    Ditolak/NO   (0)       : {label_tolak}')
print(f'    Unknown      (-1)      : {label_unknwn}')
print()
print(f'  Rata-rata word count     : {avg_wc:,.0f} kata/dokumen')
print()
print('  File output:')
print(f'    {PATH_CSV_OUT}')
print(f'    {PATH_JSON_OUT}')
print('=' * 62)
print()

# Preview sesuai contoh kolom di soal
cols_preview = ['case_id', 'no_perkara', 'tanggal', 'ringkasan_fakta', 'pasal', 'pihak', 'argumen_hukum', 'amar_lengkap', 'word_count', 'top_terms', 'label_putusan', 'text_full']
print('Preview 3 baris pertama (kolom sesuai soal):')
df_cases[cols_preview].head(3)

  RINGKASAN TAHAP 2 — CASE REPRESENTATION
  Total kasus              : 54
  Jumlah kolom CSV         : 16

  Ekstraksi Konten:
    Ringkasan fakta        : 54/54
    Argumen hukum          : 54/54
    Amar putusan lengkap   : 54/54

  Distribusi Label Putusan:
    Dikabulkan   (1)       : 30
    Ditolak/NO   (0)       : 24
    Unknown      (-1)      : 0

  Rata-rata word count     : 12,204 kata/dokumen

  File output:
    data/processed/cases.csv
    data/processed/cases.json

Preview 3 baris pertama (kolom sesuai soal):


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,argumen_hukum,amar_lengkap,word_count,top_terms,label_putusan,text_full
0,case_001,1124/Pdt.G/2024/PN Sby,21 Oktober 2024,hal 1 dari 50 putusan nomor 1124 pdt g 2024 pn sby putusan nomor 1124 pdt g 2024 pn sby demi keadilan berdasarkan ke...,"Pasal 1365, Pasal 1320, Pasal 1338, Pasal 1321, Pasal 1320 dan",Sukariyadi Rudi Meidiyanto vs. Fiona Magdalena Yapsawaky,nya menimbang bahwa maksud dan tujuan dari gugatan dari penggugat tersebut diatas menimbang bahwa gugatan penggugat ...,mengadili dalam konpensi dalam eksepsi menolak eksepsi dari tergugat dalam pokok perkara 1 mengabulkan gugatan pengg...,16920,"fotokopi, tergugat, penggugat, nomor, tanda, diberi, tanggal, rekonpensi, invoice, kapal",1,hal 1 dari 50 putusan nomor 1124 pdt g 2024 pn sby putusan nomor 1124 pdt g 2024 pn sby demi keadilan berdasarkan ke...
1,case_002,1225/Pdt.G/2024/PN Sby,18 Nopember 2024,menimbang bahwa penggugat dengan surat gugatan tanggal 11nopember 2024 yang diterima dan didaftarkan di kepaniteraan...,"Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332",Kandar Panjaitan vs. Johan Sugiharto Se,menimbang bahwa maksud dan tujuan gugatan penggugat yang padapokoknya adalah sebagaimana tersebut di atas dalam konp...,mengadili dalam konpensi dalam eksepsi menerima eksepsi dari tergugat dalam pokok perkara menyatakan gugatan penggug...,9165,"penggugat, tergugat, bukti, selanjutnya, fotokopi, diberi, tanda, gugatan, pekerjaan, bangunan",0,p u t u s a nnomor 1225 pdt g 2024 pn sbydemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri surabaya...
2,case_003,75/Pdt.G/2025/PN Sby,16 Januari 2025,menimbang bahwa penggugat dengan surat gugatan tanggal 13 januari 2025 yang diterima dan didaftarkan di kepaniteraan...,"Pasal 1320, Pasal 1243, Pasal 1338, Pasal 118 ayat 1, Pasal 123, Pasal 227 ayat 1, Pasal 180, Pasal 1267, Pasal 1238...",Pt Multi Bangun Indonesia vs. Pt Masa Sinar Mulia,menimbang bahwa maksud gugatan penggugat adalah sebagaimana tersebut diatas menimbang bahwa atas gugatan tersebut te...,mengadili 1 menyatakan bahwa tergugat yang telah dipanggil dengan patut untuk datang menghadap dipersidangan tidak h...,6515,"tergugat, penggugat, puluh, ratus, rupiah, tanggal, juta, lima, kepada, ribu",1,hal 1 dari 21 putusan nomor 75 pdt g 2025 pn sby putusan nomor 75 pdt g 2025 pn sby demi keadilan berdasarkan ketuha...


In [107]:
print(df_cases.columns.tolist())
print()
print(df_cases[['pasal', 'pihak', 'text_full']].head(3))

['case_id', 'no_perkara', 'tanggal', 'jenis_perkara', 'pengadilan', 'pihak', 'pasal', 'ringkasan_fakta', 'argumen_hukum', 'amar_putusan', 'amar_lengkap', 'word_count', 'top_terms', 'label_putusan', 'text_full', 'cek_pihak']

                                                                                                                     pasal  \
0                                                           Pasal 1365, Pasal 1320, Pasal 1338, Pasal 1321, Pasal 1320 dan   
1  Pasal 1320 dan, Pasal 1338, Pasal 1320, Pasal 180, Pasal 191, Pasal 1243, Pasal 180 ayat 1, Pasal 191 ayat 1, Pasal 332   
2  Pasal 1320, Pasal 1243, Pasal 1338, Pasal 118 ayat 1, Pasal 123, Pasal 227 ayat 1, Pasal 180, Pasal 1267, Pasal 1238...   

                                                      pihak  \
0  Sukariyadi Rudi Meidiyanto vs. Fiona Magdalena Yapsawaky   
1                   Kandar Panjaitan vs. Johan Sugiharto Se   
2         Pt Multi Bangun Indonesia vs. Pt Masa Sinar Mulia   

                 